# 预测房价：标量回归问题示例

对应 Keras 版本：`Keras应用/第四章-神经网络入门：分类与回归/Boston房价回归.ipynb`

使用 PyTorch 完成波士顿房价预测：
- 输入：13 个房屋特征
- 输出：房价中位数（连续数值）
- 任务类型：回归问题

本实验还将学习：
1. 回归任务的数据标准化
2. 为什么输出层不加激活函数
3. 为什么用 MSE 损失和 MAE 评价
4. K 折交叉验证

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")

## 1. Boston Housing 数据集

In [ ]:
from tensorflow.keras.datasets import boston_housing

(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()

print(f"训练集特征 shape: {train_data.shape}")
print(f"测试集特征 shape: {test_data.shape}")
print(f"\n第一条训练样本的13个特征:")
print(train_data[0])
print(f"\n对应房价: {train_targets[0]} (千美元)")

## 2. 数据标准化

不同特征取值范围差别很大，需要标准化到统一尺度。

**注意**：测试集必须使用训练集的均值和标准差。

In [ ]:
# 标准化（使用训练集的均值和标准差）
mean = train_data.mean(axis=0)
std = train_data.std(axis=0)

train_data = (train_data - mean) / std
test_data = (test_data - mean) / std

# 转为 Tensor
x_train = torch.tensor(train_data, dtype=torch.float32)
y_train = torch.tensor(train_targets, dtype=torch.float32)
x_test = torch.tensor(test_data, dtype=torch.float32)
y_test = torch.tensor(test_targets, dtype=torch.float32)

print(f"标准化后第一条样本:")
print(train_data[0])

## 3. 构建回归模型

对应 Keras 版本的结构：
```
Input(13) → Dense(64, relu) → Dense(64, relu) → Dense(1)
```

**为什么输出层不加激活函数？**
- 分类问题用 sigmoid/softmax 限制输出范围
- 回归问题预测任意连续值，最后一层不加激活函数（线性输出层）

In [ ]:
def build_model():
    """
    构建 Boston 房价回归模型。
    对应 Keras 版本：Dense(64, relu) → Dense(64, relu) → Dense(1)
    """
    return nn.Sequential(
        nn.Linear(13, 64),
        nn.ReLU(),
        nn.Linear(64, 64),
        nn.ReLU(),
        nn.Linear(64, 1)  # 注意：无激活函数，输出任意实数
    )

model = build_model()
print(model)
print(f"\n总参数: {sum(p.numel() for p in model.parameters())}")

## 4. K 折交叉验证

数据量较小（训练集仅 404 条），普通验证集划分会导致结果波动大。
使用 K=4 折交叉验证获得更稳定的评估。

In [ ]:
k = 4
num_val_samples = len(train_data) // k  # 404 // 4 = 101
num_epochs = 100
all_scores = []

for i in range(k):
    print(f"正在处理第 {i + 1} 折验证...")
    
    # 切出第 i 份作为验证集
    val_start = i * num_val_samples
    val_end = (i + 1) * num_val_samples
    
    val_x = x_train[val_start:val_end]
    val_y = y_train[val_start:val_end]
    
    # 剩余 3 份拼接成训练集
    train_x = torch.cat([x_train[:val_start], x_train[val_end:]], dim=0)
    train_y = torch.cat([y_train[:val_start], y_train[val_end:]], dim=0)
    
    # 每折重新建模
    model = build_model()
    optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    
    # 训练
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss = loss_fn(output.squeeze(-1), train_y)
        loss.backward()
        optimizer.step()
    
    # 在验证集上评估 MAE
    with torch.no_grad():
        val_pred = model(val_x)
        val_mae = F.l1_loss(val_pred.squeeze(-1), val_y).item()
    
    all_scores.append(val_mae)
    print(f"  第 {i + 1} 折 MAE: {val_mae:.4f}")

print(f"\n每一折的验证 MAE: {all_scores}")
print(f"平均验证 MAE: {np.mean(all_scores):.4f}")

## 5. 观察每个 epoch 的验证误差

In [ ]:
num_epochs_detailed = 500
all_mae_histories = []

for i in range(k):
    print(f"正在处理第 {i + 1} 折详细验证...")
    
    val_start = i * num_val_samples
    val_end = (i + 1) * num_val_samples
    
    val_x = x_train[val_start:val_end]
    val_y = y_train[val_start:val_end]
    train_x = torch.cat([x_train[:val_start], x_train[val_end:]], dim=0)
    train_y = torch.cat([y_train[:val_start], y_train[val_end:]], dim=0)
    
    model = build_model()
    optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    
    fold_mae_history = []
    for epoch in range(num_epochs_detailed):
        optimizer.zero_grad()
        output = model(train_x)
        loss = loss_fn(output.squeeze(-1), train_y)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            with torch.no_grad():
                val_pred = model(val_x)
                mae = F.l1_loss(val_pred.squeeze(-1), val_y).item()
                fold_mae_history.append(mae)
    
    all_mae_histories.append(fold_mae_history)

# 计算平均 MAE 历史
average_mae_history = [
    np.mean([x[i] for x in all_mae_histories if i < len(x)])
    for i in range(len(all_mae_histories[0]))
]

# 绘制
plt.figure(figsize=(8, 5))
epochs_plot = range(10, len(average_mae_history) * 10 + 1, 10)
plt.plot(epochs_plot, average_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.title("Average validation MAE per fold")
plt.grid(True)
plt.show()

## 6. 选择最终训练轮数

In [ ]:
# 观察曲线，选择合适的 epoch 数
best_epoch = np.argmin(average_mae_history) * 10 + 10
print(f"根据验证曲线，最佳 epoch 约在: {best_epoch}")

# 用全部训练集重新训练
final_model = build_model()
optimizer = torch.optim.RMSprop(final_model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(best_epoch):
    optimizer.zero_grad()
    output = final_model(x_train)
    loss = loss_fn(output.squeeze(-1), y_train)
    loss.backward()
    optimizer.step()

print(f"使用 {best_epoch} 轮在全部训练集上训练完成")

## 7. 测试集评估

In [ ]:
with torch.no_grad():
    test_pred = final_model(x_test)
    test_mse = F.mse_loss(test_pred.squeeze(-1), y_test).item()
    test_mae = F.l1_loss(test_pred.squeeze(-1), y_test).item()

print(f"测试集 MSE: {test_mse:.4f}")
print(f"测试集 MAE: {test_mae:.4f}")
print(f"（平均误差约 {test_mae:.1f} 千美元）")

## 8. 预测示例

In [ ]:
with torch.no_grad():
    predictions = final_model(x_test).squeeze(-1)

print(f"{'样本':>6} | {'预测房价':>8} | {'真实房价':>8} | 差异")
print("-" * 45)
for i in range(10):
    diff = predictions[i] - y_test[i]
    print(f"  #{i:>4} | {predictions[i]:>8.2f} | {y_test[i]:>8.2f} | {diff:+.2f}")

### 回归 vs 分类 对比

| 维度 | 分类任务 | 回归任务 |
|------|---------|---------|
| 输出层 | `sigmoid` 或 `softmax` | 无激活函数（线性输出） |
| 损失函数 | `BCELoss` / `CrossEntropyLoss` | `MSELoss` |
| 评价指标 | Accuracy | MAE (平均绝对误差) |
| 标签类型 | `long` (整数类别) | `float32` (连续值) |
| 预测 | `argmax` | 直接使用输出值 |